# Prueba de modelos generados

Pasos seguidos para generar un test que comprueb el funcionamiento de nuestra predicción
1. Descargamos los datos desde NHANES  [✅]
2. Reetiquetar su codigo por sus respectivos nombres [✅]
3. Establecer parametos para cada enfermedad, simulando un etiquetado
4. Comprobar con los modelos si clasifico correctamente los casos

In [1]:
# Importamos librerias 
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests
from tqdm import tqdm
import joblib
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score


In [ ]:
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"
FILES = [
    "P_DEMO.xpt",      # Demografía (edad, sexo, etc.)
    "P_CBC.xpt",       # Hemograma completo (plaquetas, leucocitos, glóbulos rojos, etc.)
    "P_BIOPRO.xpt",    # Bioquímica (ALT, AST, Creatinina, Glucosa, Colesterol total, etc.)
    "P_TCHOL.xpt",     # Colesterol total
    "P_HDL.xpt",       # HDL
    "P_TRIGLY.xpt",    # Triglicéridos + LDL
    "P_GHB.xpt",       # HbA1c
    "P_INS.xpt",       # Insulina
    "P_HSCRP.xpt",     # Proteína C Reactiva
    "P_GLU.xpt",       # Glucosa en ayunas
    "P_BMX.xpt",       # BMI, peso, talla
    "P_BPXO.xpt",      # Presión arterial + frecuencia cardíaca
]

OUTPUT_FOLDER = "data_test"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ================== DESCARGA ==================
def download_file(filename):
    url = BASE_URL + filename
    filepath = os.path.join(OUTPUT_FOLDER, filename)
    if os.path.exists(filepath):
        print(f"{filename} ya existe")
        return filepath
    print(f"Descargando {filename}...")
    response = requests.get(url, stream=True)
    with open(filepath, "wb") as f:
        for chunk in tqdm(response.iter_content(chunk_size=8192), desc=filename):
            f.write(chunk)
    return filepath

print("Iniciando descarga de archivos NHANES...\n")
file_paths = [download_file(f) for f in FILES]

# ================== LECTURA Y UNIÓN ==================
print("\nLeyendo y uniendo archivos...")

dfs = {}
for filepath in file_paths:
    filename = os.path.basename(filepath)
    df = pd.read_sas(filepath, format='xport', encoding='utf-8')
    dfs[filename] = df
    print(f"✓ {filename} → {df.shape[0]:,} filas, {df.shape[1]} columnas")

# Unir todo por SEQN
merged = dfs["P_DEMO.xpt"].copy()

for name, df in dfs.items():
    if name != "P_DEMO.xpt":
        merged = pd.merge(merged, df, on='SEQN', how='inner')

print(f"\nDataFrame final: {merged.shape[0]:,} pacientes × {merged.shape[1]} columnas")

# Guardar resultados
merged.to_csv(os.path.join(OUTPUT_FOLDER, "NHANES_2017-2020_completo.csv"), index=False)



'\nBASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"\nFILES = [\n    "P_DEMO.xpt",      # Demografía (edad, sexo, etc.)\n    "P_CBC.xpt",       # Hemograma completo (plaquetas, leucocitos, glóbulos rojos, etc.)\n    "P_BIOPRO.xpt",    # Bioquímica (ALT, AST, Creatinina, Glucosa, Colesterol total, etc.)\n    "P_TCHOL.xpt",     # Colesterol total\n    "P_HDL.xpt",       # HDL\n    "P_TRIGLY.xpt",    # Triglicéridos + LDL\n    "P_GHB.xpt",       # HbA1c\n    "P_INS.xpt",       # Insulina\n    "P_HSCRP.xpt",     # Proteína C Reactiva\n    "P_GLU.xpt",       # Glucosa en ayunas\n    "P_BMX.xpt",       # BMI, peso, talla\n    "P_BPXO.xpt",      # Presión arterial + frecuencia cardíaca\n]\n\nOUTPUT_FOLDER = "data_test"\nos.makedirs(OUTPUT_FOLDER, exist_ok=True)\n\n# ================== DESCARGA ==================\ndef download_file(filename):\n    url = BASE_URL + filename\n    filepath = os.path.join(OUTPUT_FOLDER, filename)\n    if os.path.exists(filepath):\n        p

In [3]:
data_NHANES = pd.read_csv('data_test/NHANES_2017-2020_completo.csv')

In [4]:
data_NHANES.head()

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,DMDBORN4,...,BPAOCSZ,BPXOSY1,BPXODI1,BPXOSY2,BPXODI2,BPXOSY3,BPXODI3,BPXOPLS1,BPXOPLS2,BPXOPLS3
0,109264.0,66.0,2.0,2.0,13.0,NaN,1.0,1.0,2.0,1.0,...,3.0,109.0,67.0,109.0,68.0,106.0,66.0,94.0,95.0,91.0
1,109271.0,66.0,2.0,1.0,49.0,NaN,3.0,3.0,2.0,1.0,...,4.0,102.0,65.0,108.0,68.0,111.0,68.0,73.0,71.0,70.0
2,109274.0,66.0,2.0,1.0,68.0,NaN,5.0,7.0,1.0,1.0,...,3.0,138.0,70.0,132.0,69.0,132.0,71.0,NaN,NaN,NaN
3,109277.0,66.0,2.0,2.0,12.0,NaN,1.0,1.0,1.0,1.0,...,3.0,104.0,58.0,102.0,54.0,101.0,53.0,67.0,69.0,77.0
4,109282.0,66.0,2.0,1.0,76.0,NaN,3.0,3.0,2.0,1.0,...,4.0,141.0,77.0,137.0,71.0,140.0,70.0,55.0,55.0,55.0


In [ ]:
#Ahora tenemos que seleccionar solo los datos que necesitamos para evaluar el modelo
# Para esto, vamos a seleccionar las siguientes columnas:

# Equivalencias en codigos NHANES de columnas necesarias:

data_NHANES['Heart Rate'] = data_NHANES[['BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3']].mean(axis=1)

equivalencias = {
    'LBXPLTSI': 'Platelets',
    'LBXWBCSI': 'White Blood Cells',
    'LBXRBCSI': 'Red Blood Cells',
    'LBXHCT': 'Hematocrit',
    'LBXMCVSI': 'Mean Corpuscular Volume',
    'LBXMCHSI': 'Mean Corpuscular Hemoglobin',
    'LBXMC': 'Mean Corpuscular Hemoglobin Concentration',
    'LBDHDD': 'HDL Cholesterol',
    'LBXSATSI': 'ALT',
}


features = [
    'Platelets',
    'White Blood Cells',
    'Red Blood Cells',
    'Hematocrit',
    'Mean Corpuscular Volume',
    'Mean Corpuscular Hemoglobin',
    'Mean Corpuscular Hemoglobin Concentration',
    'HDL Cholesterol',
    'ALT',
    'Heart Rate'
]  

# Reetiquetamos las columnas
data_NHANES.rename(columns=equivalencias, inplace=True)
# Seleccionamos solo las columnas necesarias
data_NHANES = data_NHANES[features]



In [ ]:

# Seleccionamos solo las columnas necesarias
data_NHANES = data_NHANES[features]

# Guardamos el nuevo dataframe con las columnas reetiquetadas
data_NHANES.to_csv('data_test/NHANES_2017-2020_reetiquetado.csv', index=False)